# Phase 1 — Eval Harness Calibration

Three experiments, in order:

1. **Dataset audit** — is the golden set structurally sound?
2. **No-retrieval baseline** — what does the bare LLM already know?
3. **Judge calibration** — does the LLM judge agree with human labels?

Models per [ADR 0001](../../doc/adr/0001-model-choices.md): generation = Haiku 4.5,
judge = Sonnet 4.6, both on Bedrock. Conclusions graduate to `README.md`,
then to ADRs. This notebook is evidence, not documentation.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import boto3
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.parent
GOLDEN_PATH = REPO_ROOT / "data" / "golden_dataset.jsonl"
RUBRIC_PATH = REPO_ROOT / "doc" / "judge-rubric.md"
BASELINE_PATH = Path("baseline_answers.json")
HUMAN_LABELS_PATH = Path("human_labels.json")

GENERATION_MODEL = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
JUDGE_MODEL = "global.anthropic.claude-sonnet-4-6"

load_dotenv(REPO_ROOT / ".env")
bedrock = boto3.client("bedrock-runtime", region_name=os.environ["AWS_REGION"])

golden = [json.loads(line) for line in GOLDEN_PATH.read_text().splitlines() if line.strip()]
rubric = RUBRIC_PATH.read_text()
print(f"{len(golden)} golden questions loaded, rubric {len(rubric)} chars")

## 1. Dataset audit

Checks: all six types covered, no `TODO` evidence left in non-seed entries,
answerable questions have quoted evidence, unanswerable/out-of-domain have none.

In [ ]:
print(Counter(q["type"] for q in golden))

seeds = [q["id"] for q in golden if q.get("status") == "seed_example"]
todos = [q["id"] for q in golden if "TODO" in json.dumps(q) and q.get("status") != "seed_example"]
print(f"seed examples still present: {seeds}")
print(f"non-seed entries with TODOs (must be empty): {todos}")

real = [q for q in golden if q.get("status") != "seed_example"]
print(f"real (non-seed) questions: {len(real)}")

## 2. No-retrieval baseline

Bare LLM, zero context. For each answerable question: does the model already
know the answer from training data?

**What to look for:** if the model nails a large share of these, the dataset
is skewed toward famous papers and questions must be rewritten toward
recent/obscure work — otherwise retrieval can never prove its value.

In [ ]:
def converse(model_id: str, prompt: str, max_tokens: int = 500) -> str:
    response = bedrock.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens},
    )
    return response["output"]["message"]["content"][0]["text"]


def bare_llm_answer(question: str) -> str:
    prompt = (
        "Answer this question about machine learning research concisely. "
        "If you do not know, say so plainly.\n\n" + question
    )
    return converse(GENERATION_MODEL, prompt)


answerable = [q for q in golden if q["expected_behavior"] == "answer" and q.get("status") != "seed_example"]
if not answerable:
    print("No real answerable questions yet — write golden questions first (replace the seeds).")
else:
    baseline = []
    for q in answerable:
        answer = bare_llm_answer(q["question"])
        baseline.append({"id": q["id"], "question": q["question"], "answer": answer})
        print(f"--- {q['id']}: {q['question']}\n{answer}\n")
    BASELINE_PATH.write_text(json.dumps(baseline, indent=2))
    print(f"wrote {len(baseline)} baseline answers to {BASELINE_PATH}")

## 3. Human labels

Hand-score the baseline answers against the rubric BEFORE running the judge —
these labels are the ground truth the judge is calibrated against.

The next cell writes a template `human_labels.json` from the baseline answers.
Fill in each `verdict` (`CORRECT` / `WRONG` / `ABSTAINED`) and a short
`reason` by hand, applying `doc/judge-rubric.md` yourself.

In [ ]:
if BASELINE_PATH.exists() and not HUMAN_LABELS_PATH.exists():
    template = [
        {"id": row["id"], "verdict": "FILL_ME", "reason": "FILL_ME"}
        for row in json.loads(BASELINE_PATH.read_text())
    ]
    HUMAN_LABELS_PATH.write_text(json.dumps(template, indent=2))
    print(f"wrote template {HUMAN_LABELS_PATH} — fill in verdicts by hand")
elif HUMAN_LABELS_PATH.exists():
    labels = json.loads(HUMAN_LABELS_PATH.read_text())
    unfilled = [l["id"] for l in labels if l["verdict"] == "FILL_ME"]
    print(f"{len(labels)} labels, {len(unfilled)} still unfilled: {unfilled}")
else:
    print("Run the baseline first.")

## 4. Judge run + disagreement analysis

Runs the judge (rubric as prompt) on the same (question, answer) pairs, then
compares against your labels. Every disagreement forces either a rubric fix
(bump version in `doc/judge-rubric.md`, re-run) or a label fix.
Iterate until agreement ≥ 0.90.

In [ ]:
def judge(question: str, answer: str, reference: str | None, evidence: list) -> dict:
    evidence_text = "\n".join(e["quote"] for e in evidence) or "(none)"
    prompt = f"""You are grading an answer to a research question. Apply these rules exactly:

{rubric}

QUESTION: {question}
REFERENCE ANSWER: {reference}
SUPPORTING EVIDENCE: {evidence_text}
CANDIDATE ANSWER: {answer}

Respond with only a JSON object: {{"verdict": "CORRECT" | "WRONG" | "ABSTAINED", "reason": "<one sentence naming the deciding rule>"}}"""
    raw = converse(JUDGE_MODEL, prompt, max_tokens=200)
    start, end = raw.find("{"), raw.rfind("}") + 1
    return json.loads(raw[start:end])


# Smoke test: one synthetic pair proves the wiring end to end
smoke = judge(
    question="What rank r does the LoRA paper use in its GPT-3 experiments?",
    answer="The paper reports results with r=4.",
    reference="r ranges from 1 to 64 in the experiments; r=4 or 8 works well",
    evidence=[{"quote": "We evaluate r in {1, 2, 4, 8, 64} on GPT-3."}],
)
print("smoke test verdict:", smoke)

In [ ]:
if HUMAN_LABELS_PATH.exists() and BASELINE_PATH.exists():
    human = {l["id"]: l for l in json.loads(HUMAN_LABELS_PATH.read_text()) if l["verdict"] != "FILL_ME"}
    by_id = {q["id"]: q for q in golden}
    disagreements = []
    judged = 0
    for row in json.loads(BASELINE_PATH.read_text()):
        if row["id"] not in human:
            continue
        q = by_id[row["id"]]
        verdict = judge(q["question"], row["answer"], q.get("reference_answer"), q.get("evidence", []))
        judged += 1
        if verdict["verdict"] != human[row["id"]]["verdict"]:
            disagreements.append({"id": row["id"], "human": human[row["id"]], "judge": verdict})
    if judged:
        agreement = 1 - len(disagreements) / judged
        print(f"agreement: {agreement:.0%} over {judged} labeled pairs")
        for d in disagreements:
            print(f"\nDISAGREEMENT {d['id']}:\n  human: {d['human']['verdict']} — {d['human']['reason']}\n  judge: {d['judge']['verdict']} — {d['judge']['reason']}")
    else:
        print("No filled labels to judge yet — fill human_labels.json first.")
else:
    print("Need baseline_answers.json and human_labels.json — run sections 2 and 3 first.")

## 5. Conclusions

> Fill in, then copy to README.md and open ADRs.

- Judge–human agreement: `___%` at rubric `v_`
- No-retrieval baseline accuracy: `___%` (n=`__` answerable questions)
- Questions rewritten as too-famous: `___`
- Rubric changes made and why: `___`